
# 01 - Data Profiling

## Project
Ubuntu Retail Group - Legacy System to Modern Analytics Platform Migration

## Objective
The purpose of this notebook is to explore and profile the four legacy CSV datasets before beginning the data migration process.

## Activities
1. Load the four source datasets.
2. Inspect the structure and schema.
3. Analyse record counts.
4. Identify missing values.
5. Investigate duplicate records.
6. Examine data inconsistencies.
7. Document initial data quality findings.

## Tools
- Databricks
- Python
- PySpark
- Spark SQL

## Data Sources
- customers_legacy_large.csv
- products_legacy_large.csv
- orders_legacy_large.csv
- transactions_legacy_large.csv

## Important
The source CSV files must remain unchanged. All cleaning and transformation will happen in the Silver layer.


In [0]:
#Define the landing zone
landing_path = "/Volumes/workspace/default/urg_data_landing"

#List all files in the landing zone
files = dbutils.fs.ls(landing_path)

#Display the file names and sizes
for file in files:
  print(f"File: {file.name} | Size: {file.size:,} bytes")

In [0]:
# Load the four legacy CSV datasets
customers_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(landing_path + "/customers_legacy_large.csv")

products_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(landing_path + "/products_legacy_large.csv")

orders_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(landing_path + "/orders_legacy_large.csv")

transactions_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(landing_path + "/transactions_legacy_large.csv")

In [0]:
# Store the DataFrames in a dictionary for easy profiling
datasets = {
    "Customers": customers_df,
    "Products": products_df,
    "Orders": orders_df,
    "Transactions": transactions_df
}

# Display the number of rows in each dataset
for name, df in datasets.items():
    print(f"{name}: {df.count():,} rows")

In [0]:
for name, df in datasets.items():
    print(f"\n{'=' * 50}")
    print(f"{name.upper()} SCHEMA")
    print("=" * 50)

    df.printSchema()

In [0]:
display(customers_df.limit(10))
display(products_df.limit(10))
display(orders_df.limit(10))
display(transactions_df.limit(10))

In [0]:
from pyspark.sql import functions as F

def profile_missing_values(df, dataset_name):
    total_rows = df.count()
    results = []

    for column in df.columns:
        missing_count = df.filter(
            F.col(column).isNull() |
            (F.trim(F.col(column).cast("string")) == "")
        ).count()

        results.append((
            column,
            missing_count,
            round((missing_count / total_rows) * 100, 2)
        ))

    return spark.createDataFrame(
        results,
        ["column_name", "missing_count", "missing_percentage"]
    ).withColumn("dataset", F.lit(dataset_name))

missing_profiles = []

for name, df in datasets.items():
    print(f"\nMissing values: {name}")
    result = profile_missing_values(df, name)
    display(result)

In [0]:
primary_keys = {
    "Customers": "customer_id",
    "Products": "product_id",
    "Orders": "order_id",
    "Transactions": "transaction_id"
}

for name, df in datasets.items():
    key = primary_keys[name]

    print(f"\n{name.upper()} DUPLICATE ANALYSIS")

    # Count exact duplicate rows
    total_rows = df.count()
    distinct_rows = df.distinct().count()
    exact_duplicates = total_rows - distinct_rows

    print(f"Total rows: {total_rows:,}")
    print(f"Distinct rows: {distinct_rows:,}")
    print(f"Exact duplicate rows: {exact_duplicates:,}")

    # Identify repeated primary keys
    duplicate_keys = (
        df.groupBy(key)
        .count()
        .filter(F.col("count") > 1)
    )

    print(f"Number of repeated primary-key values: {duplicate_keys.count():,}")

    display(duplicate_keys.orderBy(F.desc("count")))

In [0]:
columns_to_check = {
    "Customers": ["account_type", "loyalty_tier", "marketing_opt_in"],
    "Products": ["category", "sub_category", "active_flag"],
    "Orders": ["status", "channel", "priority_flag"],
    "Transactions": ["payment_method", "successful_flag"]
}

for name, columns in columns_to_check.items():
    print(f"\n{'=' * 50}")
    print(f"{name.upper()} VALUE ANALYSIS")
    print("=" * 50)

    df = datasets[name]

    for column in columns:
        print(f"\nValues in {column}:")

        display(
            df.groupBy(column)
              .count()
              .orderBy(F.desc("count"))
        )

In [0]:
columns_to_inspect = {
    "Customers": ["registration_date"],
    "Products": ["cost_price", "retail_price"],
    "Orders": ["order_date", "amount"],
    "Transactions": ["transaction_date", "amount"]
}

for name, columns in columns_to_inspect.items():
    df = datasets[name]

    print(f"\n{name.upper()} FORMAT ANALYSIS")

    for column in columns:
        print(f"\nSample values from {column}:")

        display(
            df.select(column)
              .distinct()
              .limit(20)
        )

## Initial Data Profiling Findings

This section documents the data quality issues identified during the initial
profiling of the Ubuntu Retail Group legacy datasets.

The findings will inform the transformation rules, data standardisation
decisions and quarantine strategy implemented in the Silver layer.

No source records have been modified or removed during this profiling stage.